In [1]:
import os
import re
import random
import math
import io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.autograd as autograd
from skimage.metrics import structural_similarity as ssim
from contextlib import redirect_stdout

#####################################
# (A) Basic Setup: Seed & Device
#####################################
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#####################################
# (B) Condition Extraction
#####################################
def extract_condition_from_csv(csv_path, mode="three"):
    try:
        df = pd.read_csv(csv_path)
        m1 = df['max_one_hour'].mean()   if 'max_one_hour'   in df else 0.0
        m3 = df['max_three_hour'].mean() if 'max_three_hour' in df else 0.0
        m5 = df['max_five_hour'].mean()   if 'max_five_hour'  in df else 0.0
        return float(m3) if mode=="three" else float(0.5*m1 + 0.3*m3 + 0.2*m5)
    except Exception as e:
        print(f"CSV read error ({csv_path}): {e}")
        return 0.0

#####################################
# (C) File Matching & Preprocessing
#####################################
def parse_filename(fn):
    m = re.search(r"(분해\d+번).*?(\d{4}-\d{2}-\d{2}).*?(\d{4}-\d{2}-\d{2})", fn)
    return (m.group(1), m.group(2)+"_"+m.group(3)) if m else (None,None)

def get_matched_files(input_dir, target_dir, csv_dir, mode):
    inps, tgts, cvs = os.listdir(input_dir), os.listdir(target_dir), os.listdir(csv_dir)
    out = []
    for cv in cvs:
        part, dr = parse_filename(cv)
        if not part:
            continue
        ti = [f for f in tgts if part in f and dr in f]
        ii = [f for f in inps if part in f and dr in f]
        if ti and ii:
            cond = extract_condition_from_csv(os.path.join(csv_dir, cv), mode)
            out.append({
                "분해번호": part,
                "date_range": dr,
                "input_path": os.path.join(input_dir, ii[0]),
                "target_path": os.path.join(target_dir, ti[0]),
                "cond_val": cond
            })
    return out

#####################################
# (D) Original Split
#####################################
def split_train_test_strict(lst, test_ratio=0.1):
    total = len(lst)
    ntest = int(total * test_ratio)
    grp = {}
    for it in lst:
        grp.setdefault(it["분해번호"], []).append(it)
    pool = []
    for items in grp.values():
        items = sorted(items, key=lambda x: x["cond_val"])
        if len(items) >= 3:
            mids = [items[len(items)//2], items[0], items[-1]]
            cases = ["Case1", "Case3", "Case4"]
            for it, case in zip(mids, cases):
                c = it.copy()
                c["case"] = case
                pool.append(c)
        else:
            r = random.choice(items).copy()
            r["case"] = "Case2"
            pool.append(r)
    seen = set()
    uniq = []
    for it in pool:
        key = (it["분해번호"], it["date_range"])
        if key not in seen:
            seen.add(key)
            uniq.append(it)
    bycase = {}
    for it in uniq:
        bycase.setdefault(it["case"], []).append(it)
    test = []
    for c in ("Case1", "Case3", "Case4"):
        test += random.sample(bycase.get(c, []), min(2, len(bycase.get(c, []))))
    if len(test) < ntest:
        need = ntest - len(test)
        test += random.sample(bycase.get("Case2", []), min(need, len(bycase.get("Case2", []))))
    if len(test) < ntest:
        rem = [it for it in uniq if it not in test]
        test += rem[:(ntest - len(test))]
    tid = {(it["분해번호"], it["date_range"]) for it in test}
    train = [it for it in lst if (it["분해번호"], it["date_range"]) not in tid]
    return train, test

#####################################
# (E) White Data
#####################################
def generate_white_test_set(uids, inp_dir, conds):
    out = []
    files = os.listdir(inp_dir)
    for uid in uids:
        cands = [f for f in files if uid in f]
        if not cands:
            continue
        path = os.path.join(inp_dir, cands[0])
        for c in conds:
            out.append({
                "분해번호": uid,
                "cond_val": c,
                "input_path": path,
                "white_image": Image.new("RGB", (256,256), "white")
            })
    return out

def split_white_data(lst):
    grp = {}
    for it in lst:
        grp.setdefault(it["분해번호"], []).append(it)
    tr, te = [], []
    for items in grp.values():
        t = random.choice(items)
        t["case"] = "WhiteTest"
        te.append(t)
        for it in items:
            if it is not t:
                it["case"] = "WhiteTrain"
                tr.append(it)
    return tr, te

#####################################
# (F) Dataset
#####################################
class CustomPairedDataset(Dataset):
    def __init__(self, data, tf=None, reps=1):
        self.data = data
        self.tf = tf
        self.reps = reps
        self.n = len(data)
        self.inps, self.tgts, self.conds = [], [], []
        for it in data:
            inp = Image.open(it["input_path"]).convert("RGB")
            tgt = it.get("white_image") or Image.open(it["target_path"]).convert("RGB")
            if tf:
                inp, tgt = tf(inp), tf(tgt)
            else:
                inp, tgt = transforms.ToTensor()(inp), transforms.ToTensor()(tgt)
            self.inps.append(inp)
            self.tgts.append(tgt)
            self.conds.append(it["cond_val"])
        arr = np.array(self.conds, dtype=np.float32)
        mn, mx = arr.min(), arr.max()
        self.normed = (arr - mn) / (mx - mn) if mx > mn else arr - mn

    def __len__(self):
        return self.n * self.reps

    def __getitem__(self, idx):
        i = idx % self.n
        return (
            self.inps[i],
            torch.tensor([self.normed[i]], dtype=torch.float32),
            self.tgts[i],
            self.conds[i],
            self.data[i].get("case", "Orig"),
            self.data[i]["분해번호"]
        )

#####################################
# (G) Model Definitions
#####################################
class ConditionEmbed(nn.Module):
    def __init__(self, out_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.ReLU(),
            nn.Linear(32, out_dim), nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class FiLM(nn.Module):
    def __init__(self, features, cdim):
        super().__init__()
        self.gamma = nn.Linear(cdim, features)
        self.beta  = nn.Linear(cdim, features)

    def forward(self, x, ce):
        g = self.gamma(ce).unsqueeze(2).unsqueeze(3)
        b = self.beta(ce).unsqueeze(2).unsqueeze(3)
        return g * x + b

class DownBlockFiLM(nn.Module):
    def __init__(self, ic, oc, cdim, bn=True):
        super().__init__()
        self.conv = nn.Conv2d(ic, oc, 4, 2, 1, bias=False)
        self.bn   = nn.BatchNorm2d(oc) if bn else nn.Identity()
        self.film = FiLM(oc, cdim)
        self.act  = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x, ce):
        x = self.conv(x)
        x = self.bn(x)
        return self.act(self.film(x, ce))

class UpBlockFiLM(nn.Module):
    def __init__(self, ic, oc, cdim, bn=True):
        super().__init__()
        self.deconv = nn.ConvTranspose2d(ic, oc, 4, 2, 1, bias=False)
        self.bn     = nn.BatchNorm2d(oc) if bn else nn.Identity()
        self.film   = FiLM(oc, cdim)
        self.act    = nn.ReLU(inplace=True)

    def forward(self, x, ce):
        x = self.deconv(x)
        x = self.bn(x)
        return self.act(self.film(x, ce))

class UNetGeneratorFiLM(nn.Module):
    def __init__(self, cdim=16, in_c=3, out_c=3):
        super().__init__()
        self.e1 = DownBlockFiLM(in_c,   64, cdim, bn=False)
        self.e2 = DownBlockFiLM(64,    128, cdim)
        self.e3 = DownBlockFiLM(128,   256, cdim)
        self.e4 = DownBlockFiLM(256,   512, cdim)
        self.b  = DownBlockFiLM(512,   512, cdim, bn=False)
        self.d1 = UpBlockFiLM(512,     512, cdim)
        self.d2 = UpBlockFiLM(512+512, 256, cdim)
        self.d3 = UpBlockFiLM(256+256, 128, cdim)
        self.d4 = UpBlockFiLM(128+128,  64, cdim)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(64+64, 64, 4, 2, 1, bias=False),
            nn.Conv2d(64, out_c, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, x, ce):
        e1 = self.e1(x, ce)
        e2 = self.e2(e1, ce)
        e3 = self.e3(e2, ce)
        e4 = self.e4(e3, ce)
        b  = self.b(e4, ce)
        d1 = self.d1(b, ce);   d1 = torch.cat([d1, e4], 1)
        d2 = self.d2(d1, ce);  d2 = torch.cat([d2, e3], 1)
        d3 = self.d3(d2, ce);  d3 = torch.cat([d3, e2], 1)
        d4 = self.d4(d3, ce);  d4 = torch.cat([d4, e1], 1)
        return self.final(d4)

class MinibatchDiscrimination(nn.Module):
    def __init__(self, in_f, out_f, k):
        super().__init__()
        self.T = nn.Parameter(torch.Tensor(in_f, out_f, k))
        nn.init.normal_(self.T, 0, 1)
        self.out_f = out_f
        self.k     = k

    def forward(self, x):
        M = x.matmul(self.T.view(x.size(1), -1)).view(-1, self.out_f, self.k)
        B = x.size(0)
        out = []
        for i in range(B):
            diff = torch.abs(M[i].unsqueeze(0) - M)
            out.append(torch.exp(-diff).sum(0).sum(1))
        return torch.cat([x, torch.stack(out)], 1)

class PatchDiscriminatorCondMBD(nn.Module):
    def __init__(self, in_c=6, cond_c=1, f=64, mbd_f=100, mbd_k=50):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c+cond_c, f,   4,2,1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(f,           f*2, 4,2,1), nn.BatchNorm2d(f*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(f*2,         f*4, 4,2,1), nn.BatchNorm2d(f*4), nn.LeakyReLU(0.2, inplace=True)
        )
        self.pool = nn.AdaptiveAvgPool2d((4,4))
        self.flat = nn.Flatten()
        self.mbd  = MinibatchDiscrimination(f*4*4*4, mbd_f, mbd_k)
        self.fc   = nn.Linear(f*4*4*4 + mbd_f, 1)

    def forward(self, x, cond):
        B, _, H, W = x.shape
        cmap = cond.view(B,1,1,1).expand(B,1,H,W)
        x = torch.cat([x, cmap], 1)
        x = self.conv(x)
        x = self.pool(x)
        x = self.flat(x)
        x = self.mbd(x)
        return self.fc(x)

class ConditionRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,   32, 4,2,1), nn.ReLU(True),
            nn.Conv2d(32,  64, 4,2,1), nn.ReLU(True),
            nn.Conv2d(64, 128, 4,2,1), nn.ReLU(True),
            nn.Conv2d(128,256, 4,2,1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.net(x)

#####################################
# (H) Metric Utilities
#####################################
grade_info = {
    0: ("Background", "white"),
    1: ("Grade1",    "orange"),
    2: ("Grade2",    "green"),
    3: ("Grade3",    "darkgreen"),
    4: ("Grade4",    "skyblue"),
    5: ("Grade5",    "purple"),
    6: ("Grade6",    "red"),
}

def assign_grade(image_np, white_thresh=0.95):
    H, W = image_np.shape[1], image_np.shape[2]
    gm = np.zeros((H,W), int)
    bg = np.all(image_np >= white_thresh, axis=0)
    fg_idx = np.where(~bg)
    fg_pixels = image_np[:, fg_idx[0], fg_idx[1]].T
    colors = np.array([
        [1,1,1],
        [1,0.62,0.09],
        [0,1,0],
        [0,0.54,0.34],
        [0.24,0.63,0.83],
        [0.56,0.42,0.85],
        [0.86,0.42,0.42]
    ])
    dists = np.linalg.norm(fg_pixels[:,None,:] - colors[None,:,:], axis=-1)
    assigned = np.argmin(dists, axis=1)
    gm[fg_idx] = assigned
    return gm

def compute_psnr(fake, target):
    mse = ((fake - target)**2).mean().item()
    return float('inf') if mse==0 else 20 * math.log10(1.0/math.sqrt(mse))

def compute_ssim(fake, target, win=7):
    f = fake.cpu().permute(1,2,0).numpy()
    t = target.cpu().permute(1,2,0).numpy()
    return ssim(f, t, win_size=win, channel_axis=-1, data_range=1.0)

def compute_fg_iou_and_f1(fake, target, wt=0.95):
    fm = (~torch.all(fake >= wt, 1)).cpu().numpy()
    tm = (~torch.all(target >= wt, 1)).cpu().numpy()
    inter = np.logical_and(fm, tm).sum()
    uni   = np.logical_or(fm, tm).sum()
    iou   = inter/uni if uni>0 else 0
    dice  = 2*inter/(fm.sum()+tm.sum()) if (fm.sum()+tm.sum())>0 else 0
    return iou, dice

def compute_color_iou_and_f1(fake, target, wt=0.95):
    fg = assign_grade(fake.cpu().numpy(), wt)
    tg = assign_grade(target.cpu().numpy(), wt)
    mask = (tg != 0)
    inter = np.logical_and(fg==tg, mask).sum()
    uni   = np.logical_or(mask, fg!=0).sum()
    iou   = inter/uni if uni>0 else 0
    dice  = 2*inter/(np.sum(fg!=0)+np.sum(tg!=0)) if (np.sum(fg!=0)+np.sum(tg!=0))>0 else 0
    return iou, dice

def evaluate_dataset_metrics(dataloader, netG, netCond, wt=0.95):
    """
    Evaluate dataset metrics and print per-case and overall summary,
    including per-class average IoU for each case.

    Args:
        dataloader: DataLoader for evaluation set.
        netG:        Generator network.
        netCond:     Condition embedding network.
        wt:          White threshold for binarization.

    Returns:
        A string containing the captured log output.
    """
    netG.eval()
    netCond.eval()
    buf = io.StringIO()

    with redirect_stdout(buf):
        # Initialize containers for overall metrics
        overall = {
            "psnr": [], "ssim": [],
            "iou_bin": [], "f1_bin": [],
            "iou_color": [], "f1_color": [],
            "miou": [], "fg_miou": []
        }
        # Initialize per-case storage
        group_res = {}
        # Optionally track per-class IoU overall
        overall_class_iou = {g: [] for g in grade_info}

        with torch.no_grad():
            for inp, cond, tgt, orig, case, part in dataloader:
                inp, cond, tgt = inp.to(device), cond.to(device), tgt.to(device)
                emb  = netCond(cond)
                fake = netG(inp, emb)

                for b in range(fake.size(0)):
                    f = fake[b]
                    t = tgt[b]

                    # --- 기본 지표 계산 ---
                    p    = compute_psnr(f, t)
                    s_   = compute_ssim(f, t)
                    ib, fb = compute_fg_iou_and_f1(f, t, wt)
                    ic, fc = compute_color_iou_and_f1(f, t, wt)

                    # --- 등급별 픽셀 마스크 생성 ---
                    gm_f = assign_grade(f.cpu().numpy(), wt)
                    gm_t = assign_grade(t.cpu().numpy(), wt)

                    # --- 클래스별 IoU 계산 및 저장 ---
                    grade_avg = {}
                    for g in grade_info:
                        m1    = (gm_f == g)
                        m2    = (gm_t == g)
                        inter = np.logical_and(m1, m2).sum()
                        uni   = np.logical_or(m1, m2).sum()
                        iou_val = inter / uni if uni > 0 else 0.0
                        grade_avg[g] = iou_val
                        overall_class_iou[g].append(iou_val)

                    # --- 그룹별/전체 지표 집계 ---
                    lbl = case[b]
                    grp = group_res.setdefault(lbl, {
                        **{k: [] for k in overall},
                        **{f"iou_cls_{g}": [] for g in grade_info}
                    })

                    # 전체 및 그룹에 추가
                    grp["psnr"].append(p);    overall["psnr"].append(p)
                    grp["ssim"].append(s_);   overall["ssim"].append(s_)
                    grp["iou_bin"].append(ib); overall["iou_bin"].append(ib)
                    grp["f1_bin"].append(fb); overall["f1_bin"].append(fb)
                    grp["iou_color"].append(ic); overall["iou_color"].append(ic)
                    grp["f1_color"].append(fc); overall["f1_color"].append(fc)

                    miou_all = np.mean(list(grade_avg.values()))
                    miou_fg  = np.mean([v for g,v in grade_avg.items() if g != 0])
                    grp["miou"].append(miou_all); overall["miou"].append(miou_all)
                    grp["fg_miou"].append(miou_fg); overall["fg_miou"].append(miou_fg)

                    for g, v in grade_avg.items():
                        grp[f"iou_cls_{g}"].append(v)

        # --- 그룹별 요약 출력 (케이스별) ---
        for lbl, mets in group_res.items():
            avg = {k: np.mean(v) for k, v in mets.items()}
            print(
                f"[{lbl}] PSNR={avg['psnr']:.2f}, SSIM={avg['ssim']:.4f}, "
                f"IoU_bin={avg['iou_bin']:.4f}, IoU_color={avg['iou_color']:.4f}, "
                f"mIoU_all={avg['miou']:.4f}, mIoU_fg={avg['fg_miou']:.4f}"
            )
            # 클래스별 평균 IoU
            for g in sorted(grade_info):
                name, color = grade_info[g]
                print(f"  - {name}({color}): {avg[f'iou_cls_{g}']:.4f}")
            print()

        # --- 전체 요약 출력 ---
        oavg = {k: np.mean(v) for k, v in overall.items() if k in ["psnr","ssim","miou","fg_miou"]}
        print(
            f"[Overall] PSNR={oavg['psnr']:.2f}, SSIM={oavg['ssim']:.4f}, "
            f"mIoU_all={oavg['miou']:.4f}, mIoU_fg={oavg['fg_miou']:.4f}"
        )
        for g in sorted(grade_info):
            name, color = grade_info[g]
            print(f"  - {name}({color}): {np.mean(overall_class_iou[g]):.4f}")

    netG.train()
    netCond.train()
    return buf.getvalue()



#####################################
# (I) Visualization (4 cols)
#####################################
def visualize_samples(dataset, netG, netCond, out_dir, sample_limit=None, show=False):
    os.makedirs(out_dir, exist_ok=True)
    netG.eval()
    netCond.eval()
    idxs = list(range(len(dataset))) if sample_limit is None else random.sample(range(len(dataset)), sample_limit)
    with torch.no_grad():
        for idx in idxs:
            inp, cond, tgt, orig, case, part = dataset[idx]
            inp4 = inp.unsqueeze(0).to(device)
            cd4  = cond.unsqueeze(0).to(device)
            emb  = netCond(cd4)
            fake = netG(inp4, emb)[0]
            inp_img  = (inp*0.5+0.5).permute(1,2,0).cpu().numpy()
            tgt_img  = (tgt*0.5+0.5).permute(1,2,0).cpu().numpy()
            fake_img = (fake*0.5+0.5).clamp(0,1).permute(1,2,0).cpu().numpy()
            fig, axes = plt.subplots(1,4,figsize=(16,4))
            axes[0].imshow(inp_img);  axes[0].set_title("Input");     axes[0].axis("off")
            axes[1].text(0.5,0.5,f"cond={orig}",ha="center",va="center")
            axes[1].set_title("Condition"); axes[1].axis("off")
            axes[2].imshow(tgt_img);  axes[2].set_title("Target");    axes[2].axis("off")
            axes[3].imshow(fake_img); axes[3].set_title("Generated"); axes[3].axis("off")
            fn = f"{part}_{case}_{idx}.png".replace(":", "")
            fig.savefig(os.path.join(out_dir, fn))
            if show:
                plt.show()
            plt.close(fig)
    netG.train()
    netCond.train()

#####################################
# (J) Training Loop w/ Logging, Best-model, & Graphs
#####################################
def compute_gradient_penalty(D, real, fake, cond, gp_lambda=10.0):
    alpha = torch.rand(real.size(0),1,1,1,device=device).expand_as(real)
    inter = (alpha*real + (1-alpha)*fake).detach().requires_grad_(True)
    d_inter = D(inter, cond)
    grads = autograd.grad(d_inter.sum(), inter, create_graph=True)[0]
    return ((grads.view(grads.size(0),-1).norm(2,dim=1) - 1)**2).mean() * gp_lambda

def train_multi_run(params, ds_train, ds_test, ds_wte):
    # ------------------------------------------------------------------
    # Best-tracking for overall metrics
    # ------------------------------------------------------------------
    best_psnr = -float('inf');    best_epoch_psnr = -1
    best_ssim = -float('inf');    best_epoch_ssim = -1
    best_miou_all = -float('inf');best_epoch_miou_all = -1
    best_miou_fg  = -float('inf');best_epoch_miou_fg  = -1

    # ------------------------------------------------------------------
    # Best-tracking for each grade IoU
    # ------------------------------------------------------------------
    best_iou_cls       = {g: -float('inf') for g in grade_info}
    best_epoch_iou_cls = {g: -1 for g in grade_info}

    # ------------------------------------------------------------------
    # Histories for evaluation
    # ------------------------------------------------------------------
    eval_epochs  = []
    psnr_hist    = []
    ssim_hist    = []
    mIoU_hist    = []
    fg_mIoU_hist = []

    # ------------------------------------------------------------------
    # Histories for losses
    # ------------------------------------------------------------------
    epochs_hist = []
    D_hist = []
    G_hist = []
    L1_hist = []
    C_hist = []

    # ------------------------------------------------------------------
    # Model, optimizers, and losses
    # ------------------------------------------------------------------
    netCond = ConditionEmbed(params["cond_dim"]).to(device)
    netG    = UNetGeneratorFiLM(params["cond_dim"]).to(device)
    netD    = PatchDiscriminatorCondMBD().to(device)
    netR    = ConditionRegressor().to(device)
    optG = optim.Adam(list(netG.parameters()) + list(netCond.parameters()),
                      lr=params["g_lr"], betas=(0.5,0.999))
    optD = optim.Adam(netD.parameters(), lr=params["d_lr"], betas=(0.5,0.999))
    optR = optim.Adam(netR.parameters(), lr=params["g_lr"], betas=(0.5,0.999))
    L1   = nn.L1Loss()
    MSE  = nn.MSELoss()

    lt = DataLoader(ds_train, batch_size=params["batch_size"], shuffle=True)
    lv = DataLoader(ds_test,  batch_size=params["batch_size"], shuffle=False)
    lw = DataLoader(ds_wte,   batch_size=params["batch_size"], shuffle=False)

    out_dir  = params["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    log_file = os.path.join(out_dir, "eval_logs.txt")
    loss_log = os.path.join(out_dir, "loss_logs.txt")

    # ------------------------------------------------------------------
    # Training across epochs
    # ------------------------------------------------------------------
    for epoch in range(1, params["num_epochs"] + 1):
        D_losses = []
        G_losses = []
        L1_losses = []
        C_losses = []
        loop = tqdm(lt, desc=f"Epoch {epoch}/{params['num_epochs']}")
        for inp, cond, tgt, *_ in loop:
            inp, cond, tgt = inp.to(device), cond.to(device), tgt.to(device)
            emb  = netCond(cond)
            fake = netG(inp, emb)

            # Discriminator step
            optD.zero_grad()
            real_pair = torch.cat([inp, tgt], 1)
            fake_pair = torch.cat([inp, fake.detach()], 1)
            d_loss = netD(fake_pair, cond).mean() - netD(real_pair, cond).mean()
            d_loss += compute_gradient_penalty(netD, real_pair, fake_pair, cond)
            d_loss.backward()
            optD.step()

            # Generator step
            optG.zero_grad(); optR.zero_grad()
            fake_pair2 = torch.cat([inp, fake], 1)
            g_loss = -netD(fake_pair2, cond).mean()
            l1_loss = L1(fake, tgt)
            c_loss  = MSE(netR(fake), cond)
            g_total = g_loss + params["lambda_l1"] * l1_loss + params["lambda_cond"] * c_loss
            g_total.backward()
            optG.step()
            optR.step()

            D_losses.append(d_loss.item())
            G_losses.append(g_total.item())
            L1_losses.append(l1_loss.item())
            C_losses.append(c_loss.item())
            loop.set_postfix(D=np.mean(D_losses), G=np.mean(G_losses))

        # 기록: loss histories
        epochs_hist.append(epoch)
        D_hist.append(np.mean(D_losses))
        G_hist.append(np.mean(G_losses))
        L1_hist.append(np.mean(L1_losses))
        C_hist.append(np.mean(C_losses))
        with open(loss_log, "a") as lf:
            lf.write(f"{epoch}, {D_hist[-1]:.6f}, {G_hist[-1]:.6f}, {L1_hist[-1]:.6f}, {C_hist[-1]:.6f}\n")

        # Evaluation & Visualization
        if epoch % params["eval_interval"] == 0:
            visualize_samples(ds_test,  netG, netCond, os.path.join(out_dir, f"{epoch}epoch/Test"))
            visualize_samples(ds_train, netG, netCond, os.path.join(out_dir, f"{epoch}epoch/Train"), sample_limit=2)
            visualize_samples(ds_wte,   netG, netCond, os.path.join(out_dir, f"{epoch}epoch/White"), sample_limit=2)

            test_str  = evaluate_dataset_metrics(lv, netG, netCond, wt=params["wt"])
            train_str = evaluate_dataset_metrics(lt, netG, netCond, wt=params["wt"])
            white_str = evaluate_dataset_metrics(lw, netG, netCond, wt=params["wt"])

            with open(log_file, "a") as lf:
                lf.write(f"\n=== Epoch {epoch} ===\n")
                lf.write(test_str + "\n" + train_str + "\n" + white_str + "\n")

            # --- Overall metrics 파싱 ---
            m = re.search(
                r"\[Overall\].*PSNR=([\d\.]+), SSIM=([\d\.]+), mIoU_all=([\d\.]+), mIoU_fg=([\d\.]+)",
                test_str
            )
            if m:
                curr_psnr     = float(m.group(1))
                curr_ssim     = float(m.group(2))
                curr_miou_all = float(m.group(3))
                curr_miou_fg  = float(m.group(4))

                eval_epochs.append(epoch)
                psnr_hist.append(curr_psnr)
                ssim_hist.append(curr_ssim)
                mIoU_hist.append(curr_miou_all)
                fg_mIoU_hist.append(curr_miou_fg)

                # Overall best 갱신
                if curr_psnr     > best_psnr:
                    best_psnr,    best_epoch_psnr    = curr_psnr,    epoch
                if curr_ssim     > best_ssim:
                    best_ssim,    best_epoch_ssim    = curr_ssim,    epoch
                if curr_miou_all > best_miou_all:
                    best_miou_all,best_epoch_miou_all= curr_miou_all,epoch
                if curr_miou_fg  > best_miou_fg:
                    best_miou_fg, best_epoch_miou_fg  = curr_miou_fg, epoch

                # 각 그레이드별 best IoU 갱신
                for g, (gname, _) in grade_info.items():
                    regex = rf"\s+-\s+{re.escape(gname)}\([^\)]*\):\s*([\d\.]+)"
                    m_cls = re.search(regex, test_str)
                    if m_cls:
                        val = float(m_cls.group(1))
                        if val > best_iou_cls[g]:
                            best_iou_cls[g]       = val
                            best_epoch_iou_cls[g] = epoch

    # ------------------------------------------------------------------
    # Training 완료 후: best summary 출력 및 저장
    # ------------------------------------------------------------------
    print(f"\n>>> Best PSNR:    {best_psnr:.2f} at epoch {best_epoch_psnr}")
    print(f">>> Best SSIM:    {best_ssim:.4f} at epoch {best_epoch_ssim}")
    print(f">>> Best mIoU_all:{best_miou_all:.4f} at epoch {best_epoch_miou_all}")
    print(f">>> Best fg-mIoU: {best_miou_fg:.4f} at epoch {best_epoch_miou_fg}")
    for g, (gname, _) in grade_info.items():
        print(f">>> Best IoU_{gname}: {best_iou_cls[g]:.4f} at epoch {best_epoch_iou_cls[g]}")

    with open(os.path.join(out_dir, "best_metrics.txt"), "w") as bf:
        bf.write(f"Best PSNR:    {best_psnr:.2f} at epoch {best_epoch_psnr}\n")
        bf.write(f"Best SSIM:    {best_ssim:.4f} at epoch {best_epoch_ssim}\n")
        bf.write(f"Best mIoU_all:{best_miou_all:.4f} at epoch {best_epoch_miou_all}\n")
        bf.write(f"Best fg-mIoU: {best_miou_fg:.4f} at epoch {best_epoch_miou_fg}\n")
        for g, (gname, _) in grade_info.items():
            bf.write(f"Best IoU_{gname}: {best_iou_cls[g]:.4f} at epoch {best_epoch_iou_cls[g]}\n")

    # ------------------------------------------------------------------
    # Eval & loss curves 저장
    # ------------------------------------------------------------------
    for name, hist in [
        ("PSNR", psnr_hist),
        ("SSIM", ssim_hist),
        ("mIoU_all", mIoU_hist),
        ("mIoU_fg", fg_mIoU_hist)
    ]:
        plt.figure()
        plt.plot(eval_epochs, hist)
        plt.title(f"{name} over Eval-Epochs")
        plt.xlabel("Epoch")
        plt.ylabel(name)
        plt.savefig(os.path.join(out_dir, f"{name}_curve.png"))
        plt.close()

    for name, hist in [
        ("D_loss", D_hist),
        ("G_loss", G_hist),
        ("L1_loss", L1_hist),
        ("Cond_loss", C_hist)
    ]:
        plt.figure()
        plt.plot(epochs_hist, hist)
        plt.title(f"{name} over Epochs")
        plt.xlabel("Epoch")
        plt.ylabel(name)
        plt.savefig(os.path.join(out_dir, f"{name}_curve.png"))
        plt.close()

    print(">>> Training complete. Logs, images, models, graphs saved in out_dir.")


#####################################
# (L) Main
#####################################
def main():
    params = {
        "region": "노원구",
        "condition_mode": "three",
        "precipitation_conditions": [0,5,10,15,20],
        "cond_filter_min": 20,
        "test_ratio": 0.1,
        "repetitions_original": 50,
        "repetitions_white": 30,
        "cond_dim": 16,
        "g_lr": 2e-4,
        "d_lr": 2e-4,
        "batch_size": 16,
        "num_epochs": 200,
        "eval_interval": 10,
        "lambda_l1": 100.0,
        "lambda_cond": 10.0,
        "wt": 0.95
    }
    root_in  = "/home/hwseo/cGan/졸작데이터/테스트4번"
    root_tgt = "/home/hwseo/cGan/졸작데이터/1.25km/이미지파일"
    root_csv = "/home/hwseo/cGan/졸작데이터/1.25km/12_02_1.25km_test1"
    region   = params["region"]
    inp_dir  = os.path.join(root_in,  region)
    tgt_dir  = os.path.join(root_tgt, region)
    csv_dir  = os.path.join(root_csv, region)

    matched = get_matched_files(inp_dir, tgt_dir, csv_dir, params["condition_mode"])
    matched = [m for m in matched if m["cond_val"] >= params["cond_filter_min"]]
    train_items, test_items = split_train_test_strict(matched, params["test_ratio"])
    uids = list({m["분해번호"] for m in matched})
    white_all = generate_white_test_set(uids, inp_dir, params["precipitation_conditions"])
    w_train, w_test = split_white_data(white_all)

    print(f"▶ Original items: {len(train_items)}")
    print(f"▶ Original after replication: {len(train_items) * params['repetitions_original']}")
    print(f"▶ WhiteTrain items: {len(w_train)}")
    print(f"▶ WhiteTrain after replication: {len(w_train) * params['repetitions_white']}")
    print(f"▶ Test items: {len(test_items)}")
    print("▶ Test-case files:")
    for it in test_items:
        print("   ", it["분해번호"], it["case"], os.path.basename(it["input_path"]))

    tf = transforms.Compose([
        transforms.Resize((256,256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    ds_train = CustomPairedDataset(train_items, tf, params["repetitions_original"])
    ds_test  = CustomPairedDataset(test_items,  tf, 1)
    ds_wtr   = CustomPairedDataset(w_train,    tf, params["repetitions_white"])
    ds_wte   = CustomPairedDataset(w_test,     tf, 1)

    out_dir = f"pix2pix논문_{region}_{params['condition_mode']}"
    params["out_dir"] = out_dir
    os.makedirs(out_dir, exist_ok=True)

    train_multi_run(params, ds_train, ds_test, ds_wte)

if __name__ == "__main__":
    main()


Using device: cuda:0
▶ Original items: 59
▶ Original after replication: 2950
▶ WhiteTrain items: 80
▶ WhiteTrain after replication: 2400
▶ Test items: 6
▶ Test-case files:
    분해389번 Case1 분해389번_노원구_2020-07-28_2020-08-11.png
    분해141번 Case1 분해141번_노원구_2016-07-05_2016-07-07.png
    분해153번 Case3 분해153번_노원구_2018-08-29_2018-09-01.png
    분해143번 Case3 분해143번_노원구_2018-08-29_2018-09-01.png
    분해143번 Case4 분해143번_노원구_2022-08-08_2022-08-17.png
    분해185번 Case4 분해185번_노원구_2022-08-08_2022-08-17.png


Epoch 200/200: 100%|██████████| 185/185 [00:53<00:00,  3.47it/s, D=-2, G=-27.8]   



>>> Best PSNR:    32.60 at epoch 40
>>> Best SSIM:    0.9975 at epoch 40
>>> Best mIoU_all:0.1428 at epoch 30
>>> Best fg-mIoU: 0.0000 at epoch 10
>>> Best IoU_Background: 0.9998 at epoch 40
>>> Best IoU_Grade1: 0.0000 at epoch 10
>>> Best IoU_Grade2: 0.0000 at epoch 10
>>> Best IoU_Grade3: 0.0000 at epoch 10
>>> Best IoU_Grade4: 0.0000 at epoch 10
>>> Best IoU_Grade5: 0.0000 at epoch 10
>>> Best IoU_Grade6: 0.0000 at epoch 10
>>> Training complete. Logs, images, models, graphs saved in out_dir.
